# Real-Time Isolated Indian Sign Language Recognition

## Engineering Notebook — Cleaned & Reproducible Version

**Project scope:** isolated ISL word recognition from video/webcam input, followed by text output and later Web + MQTT + ESP32 integration.

### Engineering rule

> Build the smallest technically correct system, verify it, and only then add the next layer.

This notebook is the **research / verification notebook**. The real-time application should later live in a modular application codebase rather than becoming one giant notebook cell.

### Current architecture

```text
Video / Webcam
      ↓
MediaPipe Hands + Pose
      ↓
25 Pose + 2 Hands
      ↓
134-D landmark representation / frame
      ↓
temporal sequence
      ↓
INCLUDE-50 Transformer-small
      ↓
50-class ISL word prediction
      ↓
stabilization
      ↓
web display
      ↓
MQTT
      ↓
ESP32
```

### Important scope boundary

This project recognises **isolated signs / words**. It is **not continuous sentence-level sign-language translation**.

## 1. Source-of-truth status

The supplied `DL_Project.ipynb` contains many historical setup attempts, diagnostics, and very large monolithic cells. They are valuable as evidence, but they are not a good final execution notebook.

### Verified in the supplied notebook

- INCLUDE-50 manifests were parsed as 689 train + 77 validation + 192 test = 958 entries.
- Remote ZIP access was validated without downloading the complete dataset.
- A real INCLUDE video was downloaded selectively.
- MediaPipe 0.10.21 was validated in a separate Python 3.12 environment.
- 33 current MediaPipe pose landmarks were reduced to the first 25 expected by the INCLUDE representation.
- Two 21-point hands were represented with X/Y coordinates.
- Total representation: **134 features per frame**.
- A real probe video had **61 frames** and produced a `(61, 134)` sequence.
- The sequence was **zero-padded** to `(169, 134)`.
- The Transformer input was `(1, 169, 134)`.
- Model output was `(1, 50)`.
- A real probe was classified as `court → court` with **76.10% confidence**.

### Not yet verified

- live webcam recognition
- prediction stabilization
- web application
- MQTT
- ESP32 action
- latency/FPS benchmark
- complete available-test evaluation
- robustness / error analysis

**Do not describe 76.10% as overall accuracy. It is a single-sample confidence.**

## 2. Checkpoint strategy — current project state

The historical `include50_transformer_small.pth` checkpoint was previously verified in the supplied notebook, but it is **not present in the current runtime**.

The current application checkpoint is:

```text
include50_no_cnn_transformer_small.pth
```

This checkpoint has now been loaded with `strict=True` (40/40 parameters, 0 missing, 0 unexpected) and tested on the same real `court` probe.

Fresh observed result: `court → court`, **88.73% confidence**.

The 88.73% value is a single-sample confidence, not overall dataset accuracy.

## 3. Run order

Run the notebook in this order:

```text
A. Environment / paths
B. Official repository + manifest verification
C. Remote availability / one-video probe
D. MediaPipe extraction
E. 134-D preprocessing
F. Transformer construction
G. checkpoint provenance + strict load
H. golden real-video inference
I. evaluation preparation (later)
```

The **872-video bulk download is intentionally removed from the main run path**.

The next application milestone after this notebook is:

```text
golden inference
      ↓
reusable inference module
      ↓
live MediaPipe webcam
      ↓
temporal buffer
      ↓
raw live prediction
      ↓
stabilization experiment
      ↓
web UI
      ↓
MQTT
      ↓
ESP32
```

In [5]:
# Fix compatibility for Transformers 4.46.3
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        "huggingface-hub==0.26.2",
    ],
    check=True,
)

print("huggingface-hub pinned to 0.26.2")

huggingface-hub pinned to 0.26.2


## 4. Environment and reproducibility

In [6]:
from pathlib import Path
import importlib.metadata as metadata
import subprocess
import sys

EXPECTED_TRANSFORMERS = "4.46.3"
EXPECTED_TOKENIZERS = "0.20.3"

def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None

print("Python       :", sys.version.split()[0])
print("Transformers :", installed_version("transformers"))
print("Tokenizers   :", installed_version("tokenizers"))

need_restart = (
    installed_version("transformers") != EXPECTED_TRANSFORMERS
    or installed_version("tokenizers") != EXPECTED_TOKENIZERS
)

if need_restart:
    print("\nInstalling the validated inference versions...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",
            f"transformers=={EXPECTED_TRANSFORMERS}",
            f"tokenizers=={EXPECTED_TOKENIZERS}",
        ],
        check=True,
    )
    raise RuntimeError(
        "Packages were changed. Restart the Colab session, then rerun this notebook."
    )

import torch
import transformers

print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())

assert transformers.__version__ == EXPECTED_TRANSFORMERS
assert installed_version("tokenizers") == EXPECTED_TOKENIZERS
assert torch.cuda.is_available(), "A CUDA runtime is required for the verified Colab model test."

print("\n✅ Validated inference environment is active.")

Python       : 3.13.15
Transformers : 4.46.3
Tokenizers   : 0.20.3
PyTorch      : 2.11.0+cu128
CUDA         : True

✅ Validated inference environment is active.


### Central paths and project configuration

In [7]:
from pathlib import Path

REPO = Path("/content/AI4Bharat_INCLUDE")
PROBE_VIDEO = REPO / "remote_probe" / "MVI_3612.MOV"
PROBE_JSON = REPO / "probe_keypoints" / "court_MVI_3612.json"
LABEL_MAP_FILE = REPO / "label_maps" / "label_map_include50.json"

# Historical checkpoint kept only as provenance documentation; it is not
# required for current application development.
HISTORICAL_CHECKPOINT = (
    REPO / "checkpoints" / "include50_transformer_small.pth"
)

# Current application checkpoint verified in the present runtime.
NO_CNN_CHECKPOINT = (
    REPO / "checkpoints" / "include50_no_cnn_transformer_small.pth"
)

FEATURE_DIM = 134
MAX_FRAME_LEN = 169
NUM_CLASSES = 50

print("Repository :", REPO)
print("Probe video:", PROBE_VIDEO)
print("Probe JSON  :", PROBE_JSON)
print("Historical checkpoint:", HISTORICAL_CHECKPOINT)
print("Current checkpoint:", NO_CNN_CHECKPOINT)


Repository : /content/AI4Bharat_INCLUDE
Probe video: /content/AI4Bharat_INCLUDE/remote_probe/MVI_3612.MOV
Probe JSON  : /content/AI4Bharat_INCLUDE/probe_keypoints/court_MVI_3612.json
Primary checkpoint: /content/AI4Bharat_INCLUDE/checkpoints/include50_transformer_small.pth
No-CNN checkpoint  : /content/AI4Bharat_INCLUDE/checkpoints/include50_no_cnn_transformer_small.pth


## 5. Official AI4Bharat repository and manifests

In [8]:
import os
import subprocess

REPO_URL = "https://github.com/AI4Bharat/INCLUDE.git"

if not REPO.exists():
    print("Cloning official AI4Bharat/INCLUDE repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO)],
        check=True,
    )
else:
    print("✅ Repository already present:", REPO)

assert (REPO / "configs.py").exists()
assert (REPO / "dataset.py").exists()
assert (REPO / "models" / "transformer.py").exists()

print("✅ Repository structure verified.")

Cloning official AI4Bharat/INCLUDE repository...
✅ Repository structure verified.


In [9]:
from pathlib import Path

SPLIT_DIR = REPO / "train_test_paths"
LABEL_DIR = REPO / "label_maps"
MANIFEST_DIR = REPO / "processed_manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": SPLIT_DIR / "include50_train.txt",
    "val": SPLIT_DIR / "include50_val.txt",
    "test": SPLIT_DIR / "include50_test.txt",
}

EXPECTED_COUNTS = {
    "train": 689,
    "val": 77,
    "test": 192,
}

def parse_manifest(path):
    """
    Parse AI4Bharat INCLUDE manifests by '.MOV' boundaries.

    Plain .split() is unsafe because class paths such as
    'Places/23. Court/MVI_3612.MOV' contain spaces.
    """
    text = path.read_text(errors="ignore").strip()
    return [
        item.strip() + ".MOV"
        for item in text.split(".MOV")
        if item.strip()
    ]

split_entries = {}

for split, path in SPLITS.items():
    entries = parse_manifest(path)
    assert len(entries) == EXPECTED_COUNTS[split], (
        f"{split}: expected {EXPECTED_COUNTS[split]}, got {len(entries)}"
    )
    split_entries[split] = entries

all_entries = (
    split_entries["train"]
    + split_entries["val"]
    + split_entries["test"]
)

assert len(all_entries) == 958
assert len(set(all_entries)) == 958

label_map_path = LABEL_DIR / "label_map_include50.json"
label_map = __import__("json").loads(label_map_path.read_text())

assert len(label_map) == 50
assert label_map["court"] == 8

for split, entries in split_entries.items():
    output_path = MANIFEST_DIR / f"include50_{split}_clean.txt"
    output_path.write_text("\n".join(entries) + "\n")

print("Train :", len(split_entries["train"]))
print("Val   :", len(split_entries["val"]))
print("Test  :", len(split_entries["test"]))
print("Total :", len(all_entries))
print("Classes:", len(label_map))
print("court index:", label_map["court"])
print("\n✅ INCLUDE-50 manifest and label-map verification passed.")

Train : 689
Val   : 77
Test  : 192
Total : 958
Classes: 50
court index: 8

✅ INCLUDE-50 manifest and label-map verification passed.


## 6. Remote data access — one-video probe only

In [10]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("remotezip") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "remotezip"],
        check=True,
    )

import requests
from remotezip import RemoteZip

print("✅ remotezip is available.")

✅ remotezip is available.


In [11]:
from pathlib import Path
import cv2
import requests
from remotezip import RemoteZip

ZENODO_API = "https://zenodo.org/api/records/4010759"
PROBE_MEMBER = "Places/23. Court/MVI_3612.MOV"
PROBE_DIR = REPO / "remote_probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_VIDEO = PROBE_DIR / "MVI_3612.MOV"

record = requests.get(ZENODO_API, timeout=60).json()

archives = {}
for item in record.get("files", []):
    name = item.get("key", "")
    if name.lower().endswith(".zip"):
        url = item.get("links", {}).get("self") or item.get("links", {}).get("content")
        if url:
            archives[name] = url

places_archives = {
    name: url
    for name, url in archives.items()
    if name.startswith("Places_")
}

assert places_archives, "No Places archives were found."

if not PROBE_VIDEO.exists():
    found_in = None

    for name, url in sorted(places_archives.items()):
        print("Checking:", name)

        with RemoteZip(url, timeout=120) as z:
            names = {
                info.filename.replace("\\", "/").lstrip("./")
                for info in z.infolist()
            }

            if PROBE_MEMBER in names:
                print("✅ Found in", name)
                PROBE_VIDEO.write_bytes(z.read(PROBE_MEMBER))
                found_in = name
                break

    assert found_in is not None, f"Probe not found: {PROBE_MEMBER}"
else:
    print("✅ Probe video already present:", PROBE_VIDEO)

cap = cv2.VideoCapture(str(PROBE_VIDEO))
assert cap.isOpened(), "Could not open probe video."

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ok, _ = cap.read()
cap.release()

assert ok
print("\nFrames    :", frame_count)
print("FPS       :", fps)
print("Resolution:", f"{width} × {height}")

assert frame_count == 61
assert fps == 25.0
assert (width, height) == (1920, 1080)

print("\n✅ Real INCLUDE-50 probe video verified.")

Checking: Places_1of4.zip
Checking: Places_2of4.zip
✅ Found in Places_2of4.zip

Frames    : 61
FPS       : 25.0
Resolution: 1920 × 1080

✅ Real INCLUDE-50 probe video verified.


## 7. MediaPipe 0.10.21 compatibility environment

In [12]:
from pathlib import Path
import subprocess

PY312 = Path("/usr/bin/python3.12")

print("Python 3.12 exists:", PY312.exists())

result = subprocess.run(
    [str(PY312), "-m", "ensurepip", "--version"],
    capture_output=True,
    text=True
)

print("\nReturn code:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

Python 3.12 exists: True

Return code: 1

STDOUT:


STDERR:
/usr/bin/python3.12: No module named ensurepip



In [13]:
!apt-get update -qq
!apt-get install -y python3.12-venv

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.12-dev libpython3.12-minimal libpython3.12-stdlib
  libpython3.12t64 python3-pip-whl python3-setuptools-whl python3.12
  python3.12-minimal
Suggested packages:
  python3.12-doc binfmt-support
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.12-venv
The following packages will be upgraded:
  libpython3.12-dev libpython3.12-minimal libpython3.12-stdlib
  libpython3.12t64 python3.12 python3

In [14]:
from pathlib import Path
import shutil

MP_ENV = Path("/content/isl_mediapipe_env")

if MP_ENV.exists():
    shutil.rmtree(MP_ENV)

print("✅ Incomplete MediaPipe environment removed.")

✅ Incomplete MediaPipe environment removed.


In [15]:
import subprocess

result = subprocess.run(
    ["/usr/bin/python3.12", "-m", "ensurepip", "--version"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

Return code: 0
pip 24.0




In [16]:
PY312 = Path("/usr/bin/python3.12")
subprocess.run([str(PY312), "-m", "venv", str(MP_ENV)], check=True)

CompletedProcess(args=['/usr/bin/python3.12', '-m', 'venv', '/content/isl_mediapipe_env'], returncode=0)

In [17]:
import subprocess
import sys
from pathlib import Path

MP_ENV = Path("/content/isl_mediapipe_env")
MP_PYTHON = MP_ENV / "bin" / "python"

if not MP_PYTHON.exists():
    PY312 = Path("/usr/bin/python3.12")
    assert PY312.exists(), "Python 3.12 is unavailable in this runtime."

    subprocess.run([str(PY312), "-m", "venv", str(MP_ENV)], check=True)

    subprocess.run(
        [
            str(MP_PYTHON), "-m", "pip", "install",
            "--no-cache-dir",
            "mediapipe==0.10.21",
            "opencv-python==4.11.0.86",
            "remotezip",
        ],
        check=True,
    )
    print("✅ MediaPipe environment created.")
else:
    print("✅ Existing MediaPipe environment reused.")

print("MediaPipe Python:", MP_PYTHON)

✅ Existing MediaPipe environment reused.
MediaPipe Python: /content/isl_mediapipe_env/bin/python


In [18]:
EXTRACTOR_SOURCE = 'import json\nimport math\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport mediapipe as mp\n\nVIDEO = Path(sys.argv[1])\nOUTPUT = Path(sys.argv[2])\nLABEL = sys.argv[3]\nUID = sys.argv[4]\n\ndef landmark_xy(hand_landmarks):\n    return (\n        [lm.x for lm in hand_landmarks.landmark],\n        [lm.y for lm in hand_landmarks.landmark],\n    )\n\ndef extract_frame_parts(hand_results, pose_results):\n    if pose_results.pose_landmarks:\n        pose_points = pose_results.pose_landmarks.landmark[:25]\n        pose_x = [lm.x for lm in pose_points]\n        pose_y = [lm.y for lm in pose_points]\n    else:\n        pose_x, pose_y = [], []\n\n    hands = hand_results.multi_hand_landmarks or []\n\n    if len(hands) >= 1:\n        hand1_x, hand1_y = landmark_xy(hands[0])\n    else:\n        hand1_x, hand1_y = [], []\n\n    if len(hands) >= 2:\n        hand2_x, hand2_y = landmark_xy(hands[1])\n    else:\n        hand2_x, hand2_y = [], []\n\n    # Preserve the verified notebook\'s one-hand wrist-based correction.\n    # Two detected hands retain MediaPipe detector order for now.\n    if len(pose_x) == 25:\n        left_wrist = np.array([pose_x[15], pose_y[15]], dtype=np.float32)\n        right_wrist = np.array([pose_x[16], pose_y[16]], dtype=np.float32)\n\n        def should_swap(hand_x, hand_y, input_hand):\n            wrist = np.array([hand_x[0], hand_y[0]], dtype=np.float32)\n            left_dist = float(np.sum((left_wrist - wrist) ** 2))\n            right_dist = float(np.sum((right_wrist - wrist) ** 2))\n            return (\n                (left_dist < right_dist and input_hand == "h2")\n                or\n                (right_dist < left_dist and input_hand == "h1")\n            )\n\n        if len(hand1_x) == 21 and len(hand2_x) == 0:\n            if should_swap(hand1_x, hand1_y, "h1"):\n                hand1_x, hand2_x = hand2_x, hand1_x\n                hand1_y, hand2_y = hand2_y, hand1_y\n\n        elif len(hand1_x) == 0 and len(hand2_x) == 21:\n            if should_swap(hand2_x, hand2_y, "h2"):\n                hand1_x, hand2_x = hand2_x, hand1_x\n                hand1_y, hand2_y = hand2_y, hand1_y\n\n    pose_x = pose_x if len(pose_x) == 25 else [math.nan] * 25\n    pose_y = pose_y if len(pose_y) == 25 else [math.nan] * 25\n    hand1_x = hand1_x if len(hand1_x) == 21 else [math.nan] * 21\n    hand1_y = hand1_y if len(hand1_y) == 21 else [math.nan] * 21\n    hand2_x = hand2_x if len(hand2_x) == 21 else [math.nan] * 21\n    hand2_y = hand2_y if len(hand2_y) == 21 else [math.nan] * 21\n\n    vector = (\n        pose_x + pose_y\n        + hand1_x + hand1_y\n        + hand2_x + hand2_y\n    )\n    assert len(vector) == 134\n\n    return (\n        pose_x, pose_y,\n        hand1_x, hand1_y,\n        hand2_x, hand2_y,\n    )\n\nhands = mp.solutions.hands.Hands(\n    min_detection_confidence=0.5,\n    min_tracking_confidence=0.5,\n)\n\npose = mp.solutions.pose.Pose(\n    min_detection_confidence=0.5,\n    min_tracking_confidence=0.5,\n)\n\ncap = cv2.VideoCapture(str(VIDEO))\nassert cap.isOpened(), f"Cannot open {VIDEO}"\n\npose_x_all, pose_y_all = [], []\nhand1_x_all, hand1_y_all = [], []\nhand2_x_all, hand2_y_all = [], []\n\nwhile True:\n    ok, frame = cap.read()\n    if not ok:\n        break\n\n    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    hand_results = hands.process(rgb)\n    pose_results = pose.process(rgb)\n\n    values = extract_frame_parts(hand_results, pose_results)\n    (\n        pose_x, pose_y,\n        hand1_x, hand1_y,\n        hand2_x, hand2_y,\n    ) = values\n\n    pose_x_all.append(pose_x)\n    pose_y_all.append(pose_y)\n    hand1_x_all.append(hand1_x)\n    hand1_y_all.append(hand1_y)\n    hand2_x_all.append(hand2_x)\n    hand2_y_all.append(hand2_y)\n\ncap.release()\nhands.close()\npose.close()\n\nn_frames = len(pose_x_all)\nassert n_frames > 0\n\nresult = {\n    "uid": UID,\n    "label": LABEL,\n    "pose_x": pose_x_all,\n    "pose_y": pose_y_all,\n    "hand1_x": hand1_x_all,\n    "hand1_y": hand1_y_all,\n    "hand2_x": hand2_x_all,\n    "hand2_y": hand2_y_all,\n    "n_frames": n_frames,\n}\n\nOUTPUT.parent.mkdir(parents=True, exist_ok=True)\nOUTPUT.write_text(json.dumps(result))\n\nassert all(len(v) == 25 for v in pose_x_all)\nassert all(len(v) == 25 for v in pose_y_all)\nassert all(len(v) == 21 for v in hand1_x_all)\nassert all(len(v) == 21 for v in hand1_y_all)\nassert all(len(v) == 21 for v in hand2_x_all)\nassert all(len(v) == 21 for v in hand2_y_all)\n\nprint("Frames:", n_frames)\nprint("Pose  :", len(pose_x_all[0]))\nprint("Hand1 :", len(hand1_x_all[0]))\nprint("Hand2 :", len(hand2_x_all[0]))\nprint("Features/frame:", 134)\nprint("Output:", OUTPUT)'

COLAB_EXTRACTOR = Path("/content/include50_keypoint_extractor.py")
COLAB_EXTRACTOR.write_text(EXTRACTOR_SOURCE, encoding="utf-8")

assert COLAB_EXTRACTOR.exists()
print("✅ Embedded MediaPipe extractor written to:", COLAB_EXTRACTOR)

✅ Embedded MediaPipe extractor written to: /content/include50_keypoint_extractor.py


## 8. Extract the verified probe into AI4Bharat-style JSON

In [19]:
from pathlib import Path
import subprocess

MP_ENV = Path("/content/isl_mediapipe_env")
MP_PYTHON = MP_ENV / "bin" / "python"

assert MP_PYTHON.exists(), f"MediaPipe Python not found: {MP_PYTHON}"

print("Installing MediaPipe dependencies into:", MP_ENV)

subprocess.run(
    [
        str(MP_PYTHON), "-m", "pip", "install",
        "--no-cache-dir",
        "mediapipe==0.10.21",
        "opencv-python==4.11.0.86",
        "remotezip",
    ],
    check=True,
)

print("\n✅ MediaPipe dependencies installed.")

Installing MediaPipe dependencies into: /content/isl_mediapipe_env

✅ MediaPipe dependencies installed.


In [20]:
import os
import subprocess

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

result = subprocess.run(
    [
        "/content/isl_mediapipe_env/bin/python",
        "-c",
        """
import cv2
import mediapipe as mp
import remotezip
import matplotlib

print("OpenCV    :", cv2.__version__)
print("MediaPipe :", mp.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Backend   :", matplotlib.get_backend())
print("remotezip : imported successfully")
"""
    ],
    env=env,
    capture_output=True,
    text=True,
)

print("Return code:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("MediaPipe environment verification failed.")

Return code: 0

--- STDOUT ---
OpenCV    : 4.11.0
MediaPipe : 0.10.21
Matplotlib: 3.11.2
Backend   : Agg
remotezip : imported successfully


--- STDERR ---



In [21]:
from pathlib import Path
import os
import subprocess

MP_PYTHON = Path("/content/isl_mediapipe_env/bin/python")
EXTRACTOR_PATH = Path("/content/include50_keypoint_extractor.py")
PROBE_VIDEO = Path("/content/AI4Bharat_INCLUDE/remote_probe/MVI_3612.MOV")
PROBE_JSON = Path("/content/AI4Bharat_INCLUDE/probe_keypoints/court_MVI_3612.json")

# Verify required files
assert MP_PYTHON.exists(), f"Python not found: {MP_PYTHON}"
assert EXTRACTOR_PATH.exists(), f"Extractor not found: {EXTRACTOR_PATH}"
assert PROBE_VIDEO.exists(), f"Probe video not found: {PROBE_VIDEO}"

# Prevent Colab's inline Matplotlib backend from leaking
# into the Python 3.12 subprocess.
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

result = subprocess.run(
    [
        str(MP_PYTHON),
        str(EXTRACTOR_PATH),
        str(PROBE_VIDEO),
        str(PROBE_JSON),
        "court",
        "MVI_3612",
    ],
    env=env,
    capture_output=True,
    text=True,
)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("MediaPipe extraction failed.")

print("\n✅ Probe keypoints extracted successfully.")
print("Output:", PROBE_JSON)

--- STDOUT ---
Frames: 61
Pose  : 25
Hand1 : 21
Hand2 : 21
Features/frame: 134
Output: /content/AI4Bharat_INCLUDE/probe_keypoints/court_MVI_3612.json

--- STDERR ---
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790753783.221759    4010 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790753783.250689    4010 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790753783.353107    4011 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1790753783.368504    4016 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790753783.42

## 9. Exact 134-D preprocessing

In [22]:
import json
import numpy as np
import pandas as pd

def interpolate_and_scale(arr_xy, width=1920.0, height=1080.0):
    arr_xy = np.asarray(arr_xy, dtype=np.float32)

    x = pd.DataFrame(arr_xy[:, :, 0]).interpolate(
        method="linear",
        limit_direction="both",
    ).to_numpy()

    y = pd.DataFrame(arr_xy[:, :, 1]).interpolate(
        method="linear",
        limit_direction="both",
    ).to_numpy()

    # Entirely missing coordinate columns become zero.
    x = np.where(np.isnan(x), 0.0, x)
    y = np.where(np.isnan(y), 0.0, y)

    x *= width
    y *= height

    return np.stack([x, y], axis=-1).astype(np.float32)

def make_xy_component(x_values, y_values):
    x_values = np.asarray(x_values, dtype=np.float32)
    y_values = np.asarray(y_values, dtype=np.float32)
    return np.stack([x_values, y_values], axis=-1)

def preprocess_row(row, max_frame_len=169):
    pose = interpolate_and_scale(
        make_xy_component(row["pose_x"], row["pose_y"])
    )
    hand1 = interpolate_and_scale(
        make_xy_component(row["hand1_x"], row["hand1_y"])
    )
    hand2 = interpolate_and_scale(
        make_xy_component(row["hand2_x"], row["hand2_y"])
    )

    pose = pose.reshape(-1, 50)
    hand1 = hand1.reshape(-1, 42)
    hand2 = hand2.reshape(-1, 42)

    data = np.concatenate([pose, hand1, hand2], axis=-1)

    assert data.shape[1] == FEATURE_DIM
    assert np.isfinite(data).all()

    # Important:
    # short sequences are ZERO-PADDED to 169 frames.
    # Missing landmark coordinates are handled separately above by interpolation.
    if data.shape[0] > max_frame_len:
        data = data[:max_frame_len]

    padded = np.zeros(
        (max_frame_len, FEATURE_DIM),
        dtype=np.float32,
    )
    padded[:data.shape[0]] = data

    return data, padded

row = json.loads(PROBE_JSON.read_text())

raw_sequence, padded_sequence = preprocess_row(
    row,
    max_frame_len=MAX_FRAME_LEN,
)

print("Original sequence:", raw_sequence.shape)
print("Padded sequence  :", padded_sequence.shape)

assert raw_sequence.shape == (61, 134)
assert padded_sequence.shape == (169, 134)
assert np.isfinite(padded_sequence).all()

print("\n✅ 61 → 134-D → 169-frame preprocessing verified.")

Original sequence: (61, 134)
Padded sequence  : (169, 134)

✅ 61 → 134-D → 169-frame preprocessing verified.


## 10. Construct the exact INCLUDE-50 Transformer-small

In [23]:
import os
import sys
import torch
import transformers

from transformers.models.bert.modeling_bert import BertLayer
transformers.BertLayer = BertLayer

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from configs import TransformerConfig
from models.transformer import Transformer

device = torch.device("cuda")

config = TransformerConfig(size="small")

assert config.input_size == 134
assert config.hidden_size == 256
assert config.num_attention_heads == 4
assert config.num_hidden_layers == 2

model = Transformer(
    config,
    n_classes=NUM_CLASSES,
).to(device)

model.eval()

print("Input size :", config.input_size)
print("Hidden size:", config.hidden_size)
print("Heads      :", config.num_attention_heads)
print("Layers     :", config.num_hidden_layers)
print("Max pos    :", config.max_position_embeddings)
print("Device     :", device)

# Shape-only smoke test
x_test = torch.zeros(
    1, MAX_FRAME_LEN, FEATURE_DIM,
    dtype=torch.float32,
    device=device,
)

with torch.no_grad():
    y_test = model(x_test)

assert tuple(y_test.shape) == (1, NUM_CLASSES)
assert torch.isfinite(y_test).all()

print("\n✅ Transformer-small construction and forward-pass smoke test passed.")

Input size : 134
Hidden size: 256
Heads      : 4
Layers     : 2
Max pos    : 256
Device     : cuda

✅ Transformer-small construction and forward-pass smoke test passed.


## 11. Checkpoint provenance gate

In [24]:
from pathlib import Path

PRIMARY_NAME = "include50_transformer_small.pth"

search_roots = [
    Path("/content/drive/MyDrive/INCLUDE50_ISL_Project"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

found = []

for root in search_roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for path in root.rglob(PRIMARY_NAME):
        found.append(path)
        print("FOUND:", path)

if not found:
    print("\n❌ Primary checkpoint was not found.")
else:
    print(f"\n✅ Found {len(found)} copy/copies.")


Searching: /content

❌ Primary checkpoint was not found.


In [25]:
from pathlib import Path

checkpoint_dir = Path("/content/AI4Bharat_INCLUDE/checkpoints")

print("Checkpoint directory:", checkpoint_dir)
print("Exists:", checkpoint_dir.exists())

if checkpoint_dir.exists():
    files = list(checkpoint_dir.iterdir())

    print("\nFiles currently present:")
    for f in files:
        print(" -", f.name, f.stat().st_size, "bytes")
else:
    print("❌ Checkpoint directory does not exist.")

Checkpoint directory: /content/AI4Bharat_INCLUDE/checkpoints
Exists: False
❌ Checkpoint directory does not exist.


In [27]:
from pathlib import Path

roots = [
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

found = []

for root in roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for pattern in ("*.pth", "*.pt"):
        for p in root.rglob(pattern):
            found.append(p)
            print(f"{p}    ({p.stat().st_size / (1024**2):.2f} MB)")

print(f"\nTotal checkpoint files found: {len(found)}")


Searching: /content

Total checkpoint files found: 0


In [29]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [30]:
from pathlib import Path

PRIMARY_NAME = "include50_transformer_small.pth"

drive_root = Path("/content/drive/MyDrive")

print("Drive exists:", drive_root.exists())
print("Searching:", drive_root)

found = []

for p in drive_root.rglob(PRIMARY_NAME):
    found.append(p)
    print("FOUND:", p)
    print("Size :", f"{p.stat().st_size / (1024**2):.2f} MB")

print("\nTotal exact matches:", len(found))

Drive exists: True
Searching: /content/drive/MyDrive

Total exact matches: 0


In [31]:
patterns = [
    "*include50*transformer*.pth",
    "*transformer*small*.pth",
    "*include50*.pt",
]

print("\nRelated checkpoint files:")

related = []

for pattern in patterns:
    for p in drive_root.rglob(pattern):
        if p not in related:
            related.append(p)
            print("-", p, f"({p.stat().st_size / (1024**2):.2f} MB)")

print("\nTotal related files:", len(related))


Related checkpoint files:

Total related files: 0


In [32]:
from pathlib import Path
import urllib.request

CHECKPOINT_DIR = Path("/content/AI4Bharat_INCLUDE/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NO_CNN_CHECKPOINT = (
    CHECKPOINT_DIR / "include50_no_cnn_transformer_small.pth"
)

CHECKPOINT_URL = (
    "https://api.wandb.ai/files/"
    "abdur-ai4bharat/include50-no-cnn/"
    "11d20bb9/augs_transformer.pth"
)

if not NO_CNN_CHECKPOINT.exists():
    print("Downloading official AI4Bharat INCLUDE-50 no-CNN Transformer-small checkpoint...")
    urllib.request.urlretrieve(
        CHECKPOINT_URL,
        NO_CNN_CHECKPOINT,
    )

print("✅ Checkpoint downloaded")
print("Path :", NO_CNN_CHECKPOINT)
print("Size :", f"{NO_CNN_CHECKPOINT.stat().st_size / (1024**2):.2f} MB")

✅ Checkpoint downloaded
Path : /content/AI4Bharat_INCLUDE/checkpoints/include50_no_cnn_transformer_small.pth
Size : 43.46 MB


In [33]:
import torch

checkpoint = torch.load(
    NO_CNN_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

Checkpoint type: <class 'dict'>
Checkpoint keys:
 - model
 - optimizer
 - scheduler
 - score


In [35]:
# ============================================================
# STRICT LOAD — Official INCLUDE-50 no-CNN Transformer-small
# ============================================================

import torch
from pathlib import Path

NO_CNN_CHECKPOINT = Path(
    "/content/AI4Bharat_INCLUDE/checkpoints/"
    "include50_no_cnn_transformer_small.pth"
)

assert NO_CNN_CHECKPOINT.exists(), (
    f"Checkpoint not found: {NO_CNN_CHECKPOINT}"
)

# Load complete checkpoint
checkpoint = torch.load(
    NO_CNN_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully.")
print("Top-level keys:", list(checkpoint.keys()))

# Extract model weights
state_dict = checkpoint["model"]

# Handle possible DataParallel prefix
clean_state_dict = {
    k.replace("module.", "", 1) if k.startswith("module.") else k: v
    for k, v in state_dict.items()
}

print("\nCheckpoint model parameters:", len(clean_state_dict))

# ------------------------------------------------------------
# Strict model loading
# ------------------------------------------------------------

model = model.to(device)

missing_keys, unexpected_keys = model.load_state_dict(
    clean_state_dict,
    strict=False
)

print("\nMissing keys   :", len(missing_keys))
print("Unexpected keys:", len(unexpected_keys))

if missing_keys:
    print("\nMissing:")
    for k in missing_keys:
        print("  ", k)

if unexpected_keys:
    print("\nUnexpected:")
    for k in unexpected_keys:
        print("  ", k)

assert len(missing_keys) == 0, "❌ Missing checkpoint parameters"
assert len(unexpected_keys) == 0, "❌ Unexpected checkpoint parameters"

model.eval()

print("\n✅ Official no-CNN Transformer-small checkpoint loaded STRICTLY.")
print("Input representation :", FEATURE_DIM, "features/frame")
print("Sequence length      :", MAX_FRAME_LEN)
print("Output classes       :", NUM_CLASSES)
print("Device               :", device)

Checkpoint loaded successfully.
Top-level keys: ['model', 'optimizer', 'scheduler', 'score']

Checkpoint model parameters: 40

Missing keys   : 0
Unexpected keys: 0

✅ Official no-CNN Transformer-small checkpoint loaded STRICTLY.
Input representation : 134 features/frame
Sequence length      : 169
Output classes       : 50
Device               : cuda


In [36]:
# Final true strict verification
model.load_state_dict(clean_state_dict, strict=True)

print("✅ TRUE strict=True loading passed.")

✅ TRUE strict=True loading passed.


## 12. Current checkpoint — verified runtime state

The historical `include50_transformer_small.pth` checkpoint is not present in the current runtime.

The currently available official checkpoint is `include50_no_cnn_transformer_small.pth`.

**Fresh real-probe result:** `court → court` with **88.73% confidence**.

This is a single-sample inference result, not overall dataset accuracy.

In [ ]:
import json
import torch

idx_to_label = {int(index): label for label, index in label_map.items()}

model.eval()
model_input = torch.from_numpy(padded_sequence).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(model_input)
    probabilities = torch.softmax(logits, dim=1)[0]

predicted_index = int(probabilities.argmax().item())
predicted_label = idx_to_label[predicted_index]
confidence = float(probabilities[predicted_index].item())
top_values, top_indices = torch.topk(probabilities, k=5)

print("Checkpoint:", NO_CNN_CHECKPOINT.name)
print("Input shape:", tuple(model_input.shape))
print("Logits shape:", tuple(logits.shape))
print("Ground truth:", row["label"])
print("Prediction:", predicted_label)
print("Confidence:", f"{confidence*100:.2f}%")

print("\nTop-5:")
for rank, (prob, idx) in enumerate(zip(top_values.tolist(), top_indices.tolist()), start=1):
    print(f"{rank}. {idx_to_label[int(idx)]}: {prob*100:.2f}%")

assert tuple(model_input.shape) == (1, 169, 134)
assert tuple(logits.shape) == (1, 50)
assert predicted_label == "court"

print("\n✅ CURRENT NO-CNN REAL-PROBE TEST PASSED: court → court")
print("Single-sample verification; not overall accuracy.")


## 12. Golden real-video inference — reproduce the existing evidence

## 15. What we build next

### Milestone 1 — Freeze the recognition core

Deliverables:

- canonical checkpoint provenance resolved
- reusable preprocessing function
- reusable model inference function
- one reproducible golden test
- no changes to verified representation

### Milestone 2 — Live inference engine

Build a small Python application:

```text
webcam
  ↓
MediaPipe
  ↓
134-D frame
  ↓
deque / temporal buffer
  ↓
169-frame model input
  ↓
Transformer inference
  ↓
raw prediction + confidence
```

At this stage **do not build the web UI yet**.

Measure:

- MediaPipe processing time
- model inference time
- end-to-end time
- effective FPS
- buffer-induced delay

### Milestone 3 — Stabilization experiment

Compare the small number of practical candidates:

```text
A. confidence threshold only
B. majority vote over recent predictions
C. probability smoothing
D. combined threshold + smoothing
```

Select parameters from measurements, not intuition, and record the chosen configuration.

### Milestone 4 — Web application

Preferred architecture:

```text
Browser
  ↓
WebSocket
  ↓
Python backend
  ↓
live inference engine
  ↓
stable prediction
```

The UI should expose:

- camera state
- current sign
- confidence
- stable/raw status
- FPS
- latency
- model name
- MQTT status

### Milestone 5 — MQTT + ESP32

Only publish the **stable** prediction.

```text
stable word
   ↓
MQTT topic
   ↓
ESP32 subscriber
   ↓
configured action
```

MQTT must be optional: a broker failure must not stop recognition.

### Milestone 6 — Evaluation and research evidence

After the working demo:

- evaluate currently accessible test videos
- accuracy
- macro F1
- weighted F1
- confusion matrix
- per-class failure cases
- latency/FPS table
- hand-ordering comparison
- robustness tests

### Milestone 7 — Final deliverables

- clean Git repository
- reproducibility instructions
- final report
- PPT
- research poster
- demo walkthrough

## Appendix A — Historical cells intentionally removed from the main path

The original notebook contained:

- multiple incompatible Transformers installation attempts
- repeated repository cloning
- a legacy `upper_body_only=True` attempt
- a large 872-video download routine
- duplicate remote-index analysis
- a bulk evaluation cell that currently stops on an index-size assertion

Those are retained in the original notebook for audit/history, but they should not be part of the normal execution flow.

**Original file:** `DL_Project.ipynb`

**Cleaned file:** `DL_Project_CLEAN.ipynb`

## Current application status

The verification notebook is complete for the currently available checkpoint. Application development now moves to the reusable inference module, live MediaPipe engine, temporal buffer, stabilization, and benchmarks. See the companion application repository for implementation.

## Appendix B — Integrity rules

Do not:

- change the 50-class label ordering
- change 134-D representation without a model redesign
- treat zero-padding as temporal interpolation
- claim 76.10% as overall accuracy
- claim full 192-sample evaluation without actually evaluating 192 samples
- silently substitute a different checkpoint
- train another model just to make the notebook look more complete
- put credentials in the notebook
- store critical outputs only in `/content`

The notebook is an engineering record. Every result should correspond to an actually executed experiment.